# Clasificación con Gemini — Texto Completo OCR

**Proyecto CIAI**

Misma clasificación que las pruebas anteriores pero usando el texto completo extraído por OCR
en lugar de Título + Resumen + Artículos.

**Modelo:** gemini-2.5-flash-lite  
**Estrategia:** few-shot con 6 ejemplos  
**Input:** Título + primeros N caracteres del texto OCR  
**Archivo de entrada:** `test_con_etiqueta_y_texto.xlsx` (sin la columna caso_ok para clasificar)


## 1. Instalación

In [1]:
!pip install -q openpyxl

## 2. Imports y configuración

In [2]:
import pandas as pd
import time
from google.colab import ai, files

MODELO = 'google/gemini-2.5-flash-lite'
PAUSA  = 3    # segundos entre llamadas

# Cuántos caracteres del texto OCR incluir en el prompt.
# 3000 es un balance entre información y límite de tokens.
# Podés subir a 5000 o 8000 si Colab lo permite sin errores.
MAX_CHARS_TEXTO = 3000

print('✅ Imports OK')
ai.list_models()

✅ Imports OK


['google/gemini-2.5-flash', 'google/gemini-2.5-flash-lite']

## 3. Prompt de clasificación

Mismo criterio y ejemplos que las pruebas anteriores.
La única diferencia es que ahora el input incluye el texto OCR.

In [3]:
PROMPT_SISTEMA = """Sos un experto en análisis de normativa legal argentina.
Tu tarea es clasificar si una norma consagra o no derechos de las víctimas.

CRITERIO:
- caso_ok = 1 (SÍ consagra derechos de víctimas): la norma establece derechos, garantías u obligaciones
  relativas a personas consideradas víctimas. Incluye mecanismos de denuncia, acompañamiento, asistencia,
  reparación/indemnización, o define organigramas de organismos directamente vinculados a derechos de víctimas.

- caso_ok = 0 (NO consagra derechos de víctimas): la norma menciona la palabra "víctima" pero legisla
  principalmente sobre otros sujetos (victimarios, operadores judiciales, testigos), o solo reorganiza
  estructuras administrativas, o usa el término como contexto sin involucrar directamente a las víctimas.

EJEMPLOS:
- Ley 1487 (Subsidio a víctimas de inundaciones) → 1. Otorga asistencia directa a víctimas.
- Decreto 9088 (Subsidio a deudos de víctimas de accidente) → 1. Reparación económica directa.
- Decreto 558 (Pago de sentencia CIDH a víctimas) → 1. Ejecuta reparación concreta a víctimas.
- Ley 340 (Código Civil) → 0. Legisla sobre responsabilidad civil general, no sobre víctimas.
- Decreto 851 (Organigrama Programa Verdad y Justicia) → 0. Solo define estructura administrativa.
- Ley 25633 (Día Nacional de la Memoria) → 0. Declaración conmemorativa sin derechos concretos.

INSTRUCCIÓN:
Respondé con el número 1 o 0 seguido de dos puntos y una oración breve explicando tu decisión.
Ejemplo: '1: La norma establece asistencia directa a víctimas de inundación.'"""

print('✅ Prompt configurado')

✅ Prompt configurado


## 4. Carga de datos

In [4]:
df = pd.read_excel('test_con_etiqueta_y_texto.xlsx')
df['Número'] = df['Número'].astype(str).str.strip()

print(f'Normas a clasificar: {len(df)}')
print(f'Columnas disponibles: {df.columns.tolist()}')

# Verificar que tiene la columna texto_pdf
assert 'texto_pdf' in df.columns, "ERROR: falta la columna texto_pdf"
textos_vacios = df['texto_pdf'].isna().sum()
print(f'Normas sin texto OCR: {textos_vacios}')

df.head(3)

Normas a clasificar: 62
Columnas disponibles: ['Tipo', 'Número', 'Añosanción', 'texto completo', 'periodos', 'periodos2', 'rango', 'presidencia', 'decadas', 'Título', 'Subtítulo', 'Resumen', 'Artículos', 'caso_ok', 'caso', 'internacional_ok', 'tema', 'subtema', 'tema2', 'subtema2', 'pol_ok1', 'pol_ok2', 'Comentarios', 'texto_pdf']
Normas sin texto OCR: 0


,Tipo,Número,Añosanción,texto completo,periodos,periodos2,rango,presidencia,decadas,Título,...,caso,internacional_ok,tema,subtema,tema2,subtema2,pol_ok1,pol_ok2,Comentarios,texto_pdf
0,Ley,5209,1907,ok,0,1,1,NaN,5,EMERGENCIA PéBLICA,...,Inundación en Málaga,0,3,31.0,NaN,NaN,1,NaN,NaN,BOLETIN OFICIAL\n\n55\n\nX\n\nArt. 20 Este gas...
1,Ley,26004,2004,ok,1,2,3,5.0,15,ACUERDOS,...,NaN,1,5,NaN,NaN,NaN,54,55.0,NaN,ACUERDOS http://servicios.infoleg.gob.ar/infol...
2,Ley,25815,2003,ok,1,2,3,5.0,15,CODIGO PENAL. Modificación del mismo. Sustitúy...,...,NaN,0,5,52.0,NaN,NaN,55,NaN,NaN,ARTICULO 2*% — La Universidad Nacional de\nChi...


## 5. Clasificación

In [5]:
def construir_prompt_usuario(row, max_chars=MAX_CHARS_TEXTO):
    titulo  = str(row.get('Título', '')).strip() if pd.notna(row.get('Título')) else ''
    resumen = str(row.get('Resumen', '')).strip() if pd.notna(row.get('Resumen')) else ''
    texto_pdf = str(row.get('texto_ocr', '')).strip() if pd.notna(row.get('texto_ocr')) else ''

    texto = f"Tipo: {row.get('Tipo', '')}\nNúmero: {row.get('Número', '')}\nTítulo: {titulo}"

    # Incluir resumen si existe (es corto y muy informativo)
    if resumen:
        texto += f"\nResumen: {resumen}"

    # Incluir texto OCR truncado
    if texto_pdf:
        texto += f"\n\nTexto de la norma (primeros {max_chars} caracteres):\n{texto_pdf[:max_chars]}"

    texto += "\n\n¿Esta norma consagra derechos de las víctimas?"
    return texto


def parsear_respuesta(respuesta):
    respuesta = str(respuesta).strip()
    if respuesta.startswith('1'):
        return 1, respuesta
    elif respuesta.startswith('0'):
        return 0, respuesta
    else:
        return -1, f"Respuesta inválida: {respuesta}"


predicciones    = []
justificaciones = []

for i, row in df.iterrows():
    prompt = PROMPT_SISTEMA + '\n\n' + construir_prompt_usuario(row)
    try:
        respuesta = ai.generate_text(prompt, model_name=MODELO)
        pred, just = parsear_respuesta(respuesta)
    except Exception as e:
        pred = -1
        just = f"Error: {e}"

    predicciones.append(pred)
    justificaciones.append(just)
    print(f"  [{i+1}/{len(df)}] {row.get('Tipo','')} {row.get('Número','')} → {pred}")
    time.sleep(PAUSA)

print('\n✅ Clasificación terminada')

  [1/62] Ley 5209 → 1
  [2/62] Ley 26004 → 0
  [3/62] Ley 25815 → 0
  [4/62] Ley 25632 → 1
  [5/62] Decreto Reglamentario 844 → 1
  [6/62] Decreto 1486 → 0
  [7/62] Ley 26690 → 1
  [8/62] Ley 2767 → 1
  [9/62] Ley 13116 → 1
  [10/62] Decreto 1318 → 1
  [11/62] Ley 14414 → 1
  [12/62] Ley 18577 → 1
  [13/62] Ley 21507 → 1
  [14/62] Decreto 606 → 0
  [15/62] Ley 26958 → 0
  [16/62] Ley 4552 → 1
  [17/62] Decreto 1982 → 0
  [18/62] Decreto 2009 → 0
  [19/62] Ley 24946 → 0
  [20/62] Decreto Reglamentario 235 → 1
  [21/62] Ley 25742 → 0
  [22/62] Ley 26378 → 1
  [23/62] Ley 27149 → 0
  [24/62] Ley 23280 → 0
  [25/62] Decreto 525 → 1
  [26/62] Decreto 231 → 0
  [27/62] Decreto 174 → 0
  [28/62] Ley 26842 → 1
  [29/62] Decreto 812 → 1
  [30/62] Ley 2637 → 0
  [31/62] Ley 27508 → 1
  [32/62] Decreto 888 → 0
  [33/62] Decreto 636 → 1
  [34/62] Ley 25616 → 1
  [35/62] Decreto 1800 → 1
  [36/62] Ley 26827 → 1
  [37/62] Ley 27402 → 1
  [38/62] Ley 25112 → 0
  [39/62] Decreto 840 → 0
  [40/62] Ley 

## 6. Resultados y guardado

In [6]:
df_resultado = df[['Tipo', 'Número', 'Añosanción', 'Título']].copy()
df_resultado['caso_ok_pred']  = predicciones
df_resultado['caso_ok_real']  = df['caso_ok'].values
df_resultado['Justificación'] = justificaciones

validos = [p for p in predicciones if p != -1]
errores = [p for p in predicciones if p == -1]

print(f'Total clasificadas : {len(validos)}/{len(df)}')
print(f'caso_ok_pred = 1   : {predicciones.count(1)}')
print(f'caso_ok_pred = 0   : {predicciones.count(0)}')
if errores:
    print(f'Errores (-1)       : {len(errores)}')

df_resultado.to_excel('resultado_gemini_texto_completo.xlsx', index=False)
print('\n✅ Guardado: resultado_gemini_texto_completo.xlsx')

df_resultado

Total clasificadas : 62/62
caso_ok_pred = 1   : 34
caso_ok_pred = 0   : 28

✅ Guardado: resultado_gemini_texto_completo.xlsx


,Tipo,Número,Añosanción,Título,caso_ok_pred,caso_ok_real,Justificación
0,Ley,5209,1907,EMERGENCIA PéBLICA,1,1,1: La norma establece socorros para las víctim...
1,Ley,26004,2004,ACUERDOS,0,0,0: La norma aprueba un acuerdo de asistencia j...
2,Ley,25815,2003,CODIGO PENAL. Modificación del mismo. Sustitúy...,0,0,0: La norma modifica el Código Penal y Aduaner...
3,Ley,25632,2002,CONVENCIONES,1,0,1: La norma aprueba una convención que busca p...
4,Decreto Reglamentario,844,2019,FONDO DE ASISTENCIA DIRECTA A VICTIMAS DE TRATA,1,0,1: La norma aprueba la reglamentación de un fo...
...,...,...,...,...,...,...,...
57,Ley,24417,1994,PROTECCION CONTRA LA VIOLENCIA FAMILIAR,1,0,1: La norma establece medidas de protección y ...
58,Ley,6285,1909,SUBSIDIO ESTATAL,1,1,1: La norma establece subsidios y auxilio dire...
59,Ley,27146,2015,JUSTICIA,0,0,0: La norma es un Código de Procedimiento Pena...
60,Ley,25763,2003,DERECHOS DEL NIÑO,1,0,1: La norma protege y establece derechos para ...


## 7. Descargar resultado

In [7]:
files.download('resultado_gemini_texto_completo.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>